In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# ── Carga ─────────────────────────────────────────────────────────────────────
customers = pd.read_csv('customers_dataset.csv')
geo       = pd.read_csv('geolocation_dataset.csv')
orders    = pd.read_csv('orders_dataset.csv')
payments  = pd.read_csv('order_payments_dataset.csv')

orders  = orders[orders['order_status'] != 'canceled']
pagos   = payments.groupby('order_id')['payment_value'].sum().reset_index()
geo_u   = geo[['geolocation_zip_code_prefix','geolocation_lat','geolocation_lng']].drop_duplicates('geolocation_zip_code_prefix')

df = (
    orders
    .merge(customers[['customer_id','customer_zip_code_prefix','customer_city','customer_state']], on='customer_id')
    .merge(pagos, on='order_id', how='left')
    .merge(geo_u, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')
)
df = df[df['geolocation_lat'].notna()].copy()

# NIVEL 1 — Top ciudades
top_ciudades = df.groupby('customer_city').agg(
    ingresos = ('payment_value', 'sum'),
    ordenes  = ('order_id', 'count')
).sort_values('ingresos', ascending=False).head(10)

# NIVEL 2 — Zonas dentro de São Paulo
sp = df[df['customer_city'] == 'sao paulo'].copy()
sp['zona'] = KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(sp[['geolocation_lat','geolocation_lng']].values)

zonas = sp.groupby('zona').agg(
    ingresos = ('payment_value', 'sum'),
    ordenes  = ('order_id', 'count'),
    lat      = ('geolocation_lat', 'mean'),
    lng      = ('geolocation_lng', 'mean')
).sort_values('ingresos', ascending=False).reset_index()

NOMBRES = {2: 'Vila Mariana / Ibirapuera', 1: 'Zona Leste (Penha)', 3: 'Zona Sur (Santo André)', 0: 'Zona Oeste (Lapa)', 4: 'Zona Norte (Santana)'}
zonas['nombre'] = zonas['zona'].map(NOMBRES)

# NIVEL 3 — Zona ganadora: centroide y punto de mayor densidad
zona_ganadora = zonas.iloc[0]['zona']
mejor_zona    = sp[sp['zona'] == zona_ganadora].copy()
w             = mejor_zona['payment_value'].fillna(0)

# Centroide ponderado por ingresos
centroide_lat = np.average(mejor_zona['geolocation_lat'], weights=w)
centroide_lng = np.average(mejor_zona['geolocation_lng'], weights=w)

# Punto de mayor densidad: subzona con más órdenes (K-Means k=3 dentro de la zona ganadora)
mejor_zona['subzona'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(mejor_zona[['geolocation_lat','geolocation_lng']].values)
subzonas = mejor_zona.groupby('subzona').agg(
    ingresos = ('payment_value', 'sum'),
    ordenes  = ('order_id', 'count'),
    lat      = ('geolocation_lat', 'mean'),
    lng      = ('geolocation_lng', 'mean')
).sort_values('ordenes', ascending=False).reset_index()

densidad_lat = subzonas.iloc[0]['lat']
densidad_lng = subzonas.iloc[0]['lng']
densidad_ord = subzonas.iloc[0]['ordenes']
densidad_ing = subzonas.iloc[0]['ingresos']

print(f"Centroide ponderado: lat={centroide_lat:.5f}, lng={centroide_lng:.5f}")
print(f"Punto mayor densidad: lat={densidad_lat:.5f}, lng={densidad_lng:.5f}")
print(f"  -> {densidad_ord} órdenes, R${densidad_ing:,.0f}")
print(f"\nZona ganadora total: {int(zonas.iloc[0]['ordenes'])} órdenes, R${zonas.iloc[0]['ingresos']:,.0f}")


Centroide ponderado: lat=-23.58857, lng=-46.64358
Punto mayor densidad: lat=-23.56036, lng=-46.65089
  -> 1797.0 órdenes, R$281,617

Zona ganadora total: 4640 órdenes, R$685,975
